### Generating the Embeddings

In [22]:
#Creating an autoencoder using Solo q data
import torch
import torch.nn as nn
import torch.optim as optim

import requests

import numpy as np
import pandas as pd

#Importing a hugging face dataset
from datasets import load_dataset, concatenate_datasets

In [2]:
#Creating all of the variables for the autoencoder
champions = [
    "Aatrox", "Ahri", "Akali", "Akshan", "Alistar", "Ambessa", "Amumu", "Anivia", 
    "Annie", "Aphelios", "Ashe", "Aurelion Sol", "Aurora", "Azir", "Bard", "Bel'Veth", 
    "Blitzcrank", "Brand", "Braum", "Briar", "Caitlyn", "Camille", "Cassiopeia", 
    "Cho'Gath", "Corki", "Darius", "Diana", "Dr. Mundo", "Draven", "Ekko", 
    "Elise", "Evelynn", "Ezreal", "Fiddlesticks", "Fiora", "Fizz", "Galio", 
    "Gangplank", "Garen", "Gnar", "Gragas", "Graves", "Gwen", "Hecarim", 
    "Heimerdinger", "Hwei", "Illaoi", "Irelia", "Ivern", "Janna", "Jarvan IV", 
    "Jax", "Jayce", "Jhin", "Jinx", "K'Sante", "Kai'Sa", "Kalista", "Karma", 
    "Karthus", "Kassadin", "Katarina", "Kayle", "Kayn", "Kennen", "Kha'Zix", 
    "Kindred", "Kled", "Kog'Maw", "LeBlanc", "Lee Sin", "Leona", "Lillia", 
    "Lissandra", "Lucian", "Lulu", "Lux", "Malphite", "Malzahar", "Maokai", 
    "Master Yi", "Mel", "Milio", "Miss Fortune", "Mordekaiser", "Morgana", 
    "Naafiri", "Nami", "Nasus", "Nautilus", "Neeko", "Nidalee", "Nilah", 
    "Nocturne", "Nunu & Willump", "Olaf", "Orianna", "Ornn", "Pantheon", 
    "Poppy", "Pyke", "Qiyana", "Quinn", "Rakan", "Rammus", "Rek'Sai", 
    "Rell", "Renata Glasc", "Renekton", "Rengar", "Riven", "Rumble", "Ryze", 
    "Samira", "Sejuani", "Senna", "Seraphine", "Sett", "Shaco", "Shen", 
    "Shyvana", "Singed", "Sion", "Sivir", "Skarner", "Smolder", "Sona", "Soraka", 
    "Swain", "Sylas", "Syndra", "Tahm Kench", "Taliyah", "Talon", "Taric", 
    "Teemo", "Thresh", "Tristana", "Trundle", "Tryndamere", "Twisted Fate", 
    "Twitch", "Udyr", "Urgot", "Varus", "Vayne", "Veigar", "Vel'Koz", "Vex", 
    "Vi", "Viego", "Viktor", "Vladimir", "Volibear", "Warwick", "Wukong", 
    "Xayah", "Xerath", "Xin Zhao", "Yasuo", "Yone", "Yorick", "Yunara", "Yuumi", 
    "Zaahen", "Zac", "Zed", "Zeri", "Ziggs", "Zilean", "Zoe", "Zyra"
]
print(len(champions)) #There are 172 champions as of 2026-03-01

172


In [3]:
#Creating the autoencoder class
class Champ2Vec(nn.Module):
    def __init__(self, num_champs, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_champs, embedding_dim)
        self.hidden = nn.Linear(embedding_dim, 256)
        self.output = nn.Linear(256, num_champs)
    
    def forward(self, four_champs):
        x = self.embedding(four_champs) #
        x = x.mean(dim=1) #Averaging the embeddings of the 4 champs
        x = torch.relu(self.hidden(x))
        x = self.output(x)
        return x

In [4]:
#Getting the high elo data from the hugging face dataset
#https://huggingface.co/datasets/gptilt/lol-basic-matches-challenger-10k 
dataset = load_dataset("gptilt/lol-basic-matches-challenger-10k", name="participants", split="region_americas")

participants/region_americas-00000.parqu(…):   0%|          | 0.00/10.8M [00:00<?, ?B/s]

participants/region_asia-00000.parquet:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

participants/region_europe-00000.parquet:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating region_americas split:   0%|          | 0/33340 [00:00<?, ? examples/s]

Generating region_asia split:   0%|          | 0/33340 [00:00<?, ? examples/s]

Generating region_europe split:   0%|          | 0/33340 [00:00<?, ? examples/s]

In [11]:
#Learing about the dataset
print(dataset.column_names)
print(len(dataset))
print(dataset[0])
print(dataset[0]["matchId"])
game_ids = set()
for i in range(len(dataset)):
    game_ids.add(dataset[i]["matchId"])
print(len(game_ids))

['allInPings', 'assistMePings', 'assists', 'ban', 'baronKills', 'basicPings', 'bountyLevel', 'champExperience', 'champLevel', 'championId', 'championName', 'championTransform', 'commandPings', 'consumablesPurchased', 'damageDealtToBuildings', 'damageDealtToObjectives', 'damageDealtToTurrets', 'damageSelfMitigated', 'dangerPings', 'deaths', 'detectorWardsPlaced', 'doubleKills', 'dragonKills', 'eligibleForProgression', 'enemyMissingPings', 'enemyVisionPings', 'firstBloodAssist', 'firstBloodKill', 'firstTowerAssist', 'firstTowerKill', 'gameEndedInEarlySurrender', 'gameEndedInSurrender', 'getBackPings', 'goldEarned', 'goldSpent', 'holdPings', 'individualPosition', 'inhibitorKills', 'inhibitorTakedowns', 'inhibitorsLost', 'item0', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6', 'itemsPurchased', 'killingSprees', 'kills', 'lane', 'largestCriticalStrike', 'largestKillingSpree', 'largestMultiKill', 'longestTimeSpentLiving', 'magicDamageDealt', 'magicDamageDealtToChampions', 'magicDamageT

In [17]:
df = pd.DataFrame(dataset)
df = df[["matchId", "championId", "teamId", "win", "lane", "pickTurn"]]

In [ ]:
blue_df = df[df['teamId'] == 100].groupby('matchId')['championId'].apply(list)
red_df = df[df['teamId'] == 200].groupby('matchId')['championId'].apply(list)

# Merge them back into one clean table
drafts_df = pd.concat([blue_df, red_df], axis=1, keys=['blue_team', 'red_team']).reset_index()
print(drafts_df.head())

          matchId                 blue_team                  red_team
0  BR1_3082061608      [13, 5, 16, 15, 142]   [24, 246, 254, 30, 111]
1  BR1_3082064950   [799, 876, 84, 67, 555]  [950, 72, 910, 119, 432]
2  BR1_3082079670  [68, 887, 777, 202, 111]    [126, 80, 517, 81, 43]
3  BR1_3082088035  [875, 28, 777, 145, 526]     [92, 141, 84, 42, 80]
4  BR1_3082088807   [126, 64, 99, 145, 111]   [887, 254, 893, 81, 12]
                         championId_blue            championId_red
matchId                                                           
BR1_3082061608      [13, 5, 16, 15, 142]   [24, 246, 254, 30, 111]
BR1_3082064950   [799, 876, 84, 67, 555]  [950, 72, 910, 119, 432]
BR1_3082079670  [68, 887, 777, 202, 111]    [126, 80, 517, 81, 43]
BR1_3082088035  [875, 28, 777, 145, 526]     [92, 141, 84, 42, 80]
BR1_3082088807   [126, 64, 99, 145, 111]   [887, 254, 893, 81, 12]


In [26]:
#Getting the name from the champion id
url = "https://ddragon.leagueoflegends.com/cdn/14.5.1/data/en_US/champion.json"
resp = requests.get(url).json()

id_to_name = {int(data['key']): name for name, data in resp['data'].items()}

for champ in drafts_df['red_team'][0]:
    print(id_to_name[champ])

Jax
Qiyana
Vi
Karthus
Nautilus


In [ ]:
#Generating the dataset for the autoencoder
def convert_to_training_data(drafts_df):
    training_data = []
    for idx, row in drafts_df.iterrows():
        total_list = row['blue_team'] + row['red_team']
        for i in total_list:
            input_champs = [champ for champ in total_list if champ != i]
            training_data.append((input_champs, i))
    return training_data

training_data = convert_to_training_data(drafts_df)

#Confirming training data is correct
for champ in training_data[9][0]:
    print(id_to_name[champ])
print(id_to_name[training_data[9][1]])

Ryze
XinZhao
Soraka
Sivir
Zoe
Jax
Qiyana
Vi
Karthus
Nautilus


In [ ]:
#Getting data from the csv file
#Getting the dataset from oracle elixir
df = pd.read_csv("2025_LoL_esports_match_data_from_OraclesElixir.csv")